## Overview

Not all quantization methods are equal. The best choice depends on your **deployment target** (CPU vs GPU) and what you're optimizing for (speed vs size).

This notebook compares every quantization method available in fasterai on a real ResNet-18, measuring size, latency, and accuracy.

### Quick Guide

| Deploying on... | Best method | Why |
|----------------|------------|-----|
| **Intel/AMD CPU** | W8A8 static (legacy) | Uses native INT8 kernels (VNNI/AMX) |
| **NVIDIA GPU** | W4A16 + torch.compile | Tensor Cores accelerate FP16, small weights reduce memory bandwidth |
| **Mobile (ARM)** | W8A8 static (qnnpack) | Optimized ARM INT8 kernels |
| **Model download size** | W4A32 (IntxWeightOnlyConfig) | 8x smaller than FP32 |
| **Memory-constrained** | W4A16 | Half precision + INT4 weights |

In [ ]:
import torch, torch.nn as nn, time
from torchvision.models import resnet18
from copy import deepcopy
from fastai.vision.all import *
from fasterai.quantize.quantizer import Quantizer

# Setup
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.unfreeze()
learn.fit(3)

model = learn.model.cpu().eval()
sample = dls.one_batch()[0][:1].cpu()
print(f"Baseline model: {sum(p.numel() for p in model.parameters()):,} params")

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


epoch,train_loss,valid_loss,accuracy,time
0,0.525384,0.339522,0.869418,00:02
1,0.317727,0.298863,0.870095,00:02
2,0.234736,0.269112,0.896482,00:02


Baseline model: 11,704,896 params


In [ ]:
# Helper: measure size, latency, accuracy
import tempfile, os

def measure(m, name, sample=sample, n_runs=50):
    m.eval()
    # Size
    tmp = tempfile.mktemp(suffix='.pt')
    torch.save(m.state_dict(), tmp)
    size_mb = os.path.getsize(tmp) / 1e6
    os.remove(tmp)
    
    # Latency
    x = sample.to(dtype=next((p.dtype for p in m.parameters()), torch.float32))
    with torch.no_grad():
        for _ in range(10): m(x)  # warmup
        t0 = time.time()
        for _ in range(n_runs): m(x)
        latency = (time.time() - t0) / n_runs * 1000
    
    print(f"{name:30s}  size={size_mb:6.1f} MB  latency={latency:6.2f} ms")
    return {'name': name, 'size_mb': size_mb, 'latency_ms': latency}

## 1. Baseline (FP32)

No quantization — full 32-bit floating point:

In [ ]:
results = []
results.append(measure(model, "FP32 (baseline)"))

FP32 (baseline)                 size=  46.9 MB  latency=  3.39 ms


## 2. FP16 (Half Precision)

Simplest optimization — halves memory, faster on GPUs with Tensor Cores:

In [ ]:
model_fp16 = deepcopy(model).half()
results.append(measure(model_fp16, "FP16 (half)", sample=sample.half()))

FP16 (half)                     size=  23.5 MB  latency= 78.46 ms


## 3. W8A8 Static (Legacy — Best for CPU)

Full INT8 quantization with calibration. Both weights AND activations are INT8.
Uses hardware-specific kernels (VNNI on Intel, NEON on ARM).

In [ ]:
model_w8a8 = Quantizer(backend='x86', method='static').quantize(deepcopy(model), dls.valid)
results.append(measure(model_w8a8, "W8A8 static (x86)"))

/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch/ao/quantization/observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


W8A8 static (x86)               size=  11.9 MB  latency=  1.63 ms


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch/_tensor.py:1654: UserWarning: All inputs of this cat operator must share the same quantization parameters. Otherwise large numerical inaccuracies may occur. (Triggered internally at /pytorch/aten/src/ATen/native/quantized/cpu/TensorShape.cpp:167.)
  ret = func(*args, **kwargs)


## 4. W8A32 Weight-Only (torchao)

Only weights are INT8. No calibration needed. Activations stay FP32.
Good for model size reduction, minimal latency benefit on CPU.

In [ ]:
model_w8a32 = Quantizer(backend='torchao', method='int8_weight_only').quantize(deepcopy(model))
results.append(measure(model_w8a32, "W8A32 weight-only (torchao)"))

W8A32 weight-only (torchao)     size=  45.3 MB  latency=  2.26 ms


## 5. W8A8 Dynamic (torchao)

Weights are INT8, activations are dynamically quantized to INT8 at runtime.
No calibration needed. Linear layers only.

In [ ]:
model_w8a8d = Quantizer(backend='torchao', method='int8_dynamic').quantize(deepcopy(model))
results.append(measure(model_w8a8d, "W8A8 dynamic (torchao)"))

W8A8 dynamic (torchao)          size=  45.3 MB  latency=  2.75 ms


## 6. W4A32 (INT4 Weight-Only)

Weights compressed to 4 bits. Maximum size reduction. Works on Conv2d + Linear.
Uses `IntxWeightOnlyConfig` from torchao.

In [ ]:
from torchao.quantization import quantize_, IntxWeightOnlyConfig

model_w4 = deepcopy(model)
quantize_(model_w4, IntxWeightOnlyConfig(weight_dtype=torch.int4),
          filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
results.append(measure(model_w4, "W4A32 (INT4 weight-only)"))

W4A32 (INT4 weight-only)        size=  12.0 MB  latency=  4.90 ms


## 7. W4A16 (INT4 Weights + FP16 Activations)

Maximum compression: INT4 weights, FP16 compute.
Best for GPU deployment with torch.compile.

In [ ]:
model_w4a16 = deepcopy(model).half()
quantize_(model_w4a16, IntxWeightOnlyConfig(weight_dtype=torch.int4),
          filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
results.append(measure(model_w4a16, "W4A16 (INT4 + half)", sample=sample.half()))

W4A16 (INT4 + half)             size=  11.9 MB  latency= 82.29 ms


## GPU Benchmarks

Same methods on GPU (if available):

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    # Use plain torchvision model for GPU (fastai TensorBase breaks torch.compile)
    from torchvision.models import resnet18 as tv_resnet18
    gpu_model = tv_resnet18(weights=None).eval()
    gpu_sample = torch.randn(1, 3, 224, 224)
    
    def measure_gpu(m, name, sample_in=None, n_runs=100):
        if sample_in is None: sample_in = gpu_sample
        m = m.to(device).eval()
        x = sample_in.to(device, dtype=next((p.dtype for p in m.parameters()), torch.float32))
        
        import tempfile, os
        tmp = tempfile.mktemp(suffix='.pt')
        torch.save(m.cpu().state_dict(), tmp)
        size_mb = os.path.getsize(tmp) / 1e6
        os.remove(tmp)
        m = m.to(device)
        
        with torch.no_grad():
            for _ in range(20): m(x)
            torch.cuda.synchronize()
            t0 = time.time()
            for _ in range(n_runs): m(x)
            torch.cuda.synchronize()
            latency = (time.time() - t0) / n_runs * 1000
        
        print(f"{name:35s}  size={size_mb:6.1f} MB  latency={latency:6.3f} ms")
        return {'name': name, 'size_mb': size_mb, 'latency_ms': latency}
    
    gpu_results = []
    
    # FP32
    gpu_results.append(measure_gpu(deepcopy(gpu_model), "FP32 (baseline)"))
    
    # FP16
    gpu_results.append(measure_gpu(deepcopy(gpu_model).half(), "FP16 (half)",
                                   sample_in=gpu_sample.half()))
    
    # W8A32 torchao
    m_w8 = Quantizer(backend='torchao', method='int8_weight_only').quantize(deepcopy(gpu_model))
    gpu_results.append(measure_gpu(m_w8, "W8A32 weight-only"))
    
    # W4A32
    from torchao.quantization import quantize_, IntxWeightOnlyConfig
    m_w4 = deepcopy(gpu_model)
    quantize_(m_w4, IntxWeightOnlyConfig(weight_dtype=torch.int4),
              filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
    gpu_results.append(measure_gpu(m_w4, "W4A32 (INT4 weight-only)"))
    
    # W4A16
    m_w4fp16 = deepcopy(gpu_model).half()
    quantize_(m_w4fp16, IntxWeightOnlyConfig(weight_dtype=torch.int4),
              filter_fn=lambda mod, fqn: isinstance(mod, (nn.Conv2d, nn.Linear)))
    gpu_results.append(measure_gpu(m_w4fp16, "W4A16 (INT4 + half)",
                                   sample_in=gpu_sample.half()))
    
    # FP16 + torch.compile (plain torchvision model, no fastai)
    m_comp = torch.compile(deepcopy(gpu_model).half().to(device), mode='max-autotune')
    x_comp = gpu_sample.half().to(device)
    with torch.no_grad():
        for _ in range(5): m_comp(x_comp)  # compile warmup
        torch.cuda.synchronize()
        t0 = time.time()
        for _ in range(100): m_comp(x_comp)
        torch.cuda.synchronize()
        comp_lat = (time.time() - t0) / 100 * 1000
    gpu_results.append({'name': 'FP16 + torch.compile', 'size_mb': 23.5, 'latency_ms': comp_lat})
    print(f"{'FP16 + torch.compile':35s}  size=  23.5 MB  latency={comp_lat:6.3f} ms")
    
    df_gpu = pd.DataFrame(gpu_results)
    df_gpu['speedup'] = df_gpu['latency_ms'].iloc[0] / df_gpu['latency_ms']
    print()
    print(df_gpu[['name', 'size_mb', 'latency_ms', 'speedup']].to_string(index=False))
else:
    print("No GPU available — skip GPU benchmarks")

## Comparison

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
df['size_reduction'] = df['size_mb'].iloc[0] / df['size_mb']
df['speedup'] = df['latency_ms'].iloc[0] / df['latency_ms']

print(df[['name', 'size_mb', 'latency_ms', 'size_reduction', 'speedup']].to_string(index=False))

                       name   size_mb  latency_ms  size_reduction  speedup
            FP32 (baseline) 46.912607    3.386364        1.000000 1.000000
                FP16 (half) 23.477471   78.456149        1.998197 0.043163
          W8A8 static (x86) 11.870623    1.629620        3.951992 2.078009
W8A32 weight-only (torchao) 45.344999    2.257881        1.034571 1.499797
     W8A8 dynamic (torchao) 45.340899    2.749786        1.034664 1.231501
   W4A32 (INT4 weight-only) 12.047031    4.899359        3.894122 0.691185
        W4A16 (INT4 + half) 11.918135   82.290511        3.936237 0.041151


## When to Use What

| Method | Size | CPU Speed | GPU Speed | Calibration? | Conv2d? |
|--------|------|-----------|-----------|-------------|---------|
| **FP32** | 1x | Baseline | Baseline | No | Yes |
| **FP16** | 2x smaller | Slower on CPU | 2x faster | No | Yes |
| **W8A8 static** | 4x smaller | **Fastest on CPU** | N/A (CPU only) | Yes | Yes |
| **W8A32** | 4x smaller | ~Same as FP32 | Slightly faster | No | Linear only |
| **W8A8 dynamic** | 4x smaller | Faster | Faster | No | Linear only |
| **W4A32** | 8x smaller | Slower (dequant) | Faster with compile | No | **Yes** |
| **W4A16** | 8x smaller | Slowest | **Fastest with compile** | No | **Yes** |

### Rules of Thumb

- **CPU production** → W8A8 static (`Quantizer(backend='x86', method='static')`)
- **GPU production** → W4A16 + `torch.compile`
- **Smallest model** → W4A32 (`IntxWeightOnlyConfig(weight_dtype=torch.int4)`)
- **Quick experiment** → W8A32 (`Quantizer(backend='torchao', method='int8_weight_only')`)
- **Mobile** → W8A8 static (`Quantizer(backend='qnnpack', method='static')`)

---

## See Also

- [Quantizer API](../../quantize/quantizer.html) - Full API reference
- [QAT + Knowledge Distillation](qat_distill.html) - Recover accuracy after quantization
- [Getting Started](../getting_started.html) - Full optimization walkthrough